# FoodNet — Supervised vs Self-Supervised Learning

This notebook trains a **custom CNN (< 10 M parameters, no pretrained weights)** to classify the
Food-251 dataset (251 classes, 100–600 images/class, uncontrolled input size), and solves the task under
**two paradigms** required by the specification:

1. **Supervised Learning (SL)** — the custom CNN is trained end-to-end with labels.
2. **Self-Supervised Learning (SSL)** — the *same* backbone is pretrained on the images **ignoring the labels**
   (SimCLR contrastive / rotation pretext), then its frozen features are classified with a **traditional classifier**
   (logistic regression / linear SVM / kNN).

Because the official test set has no public ground truth, the **validation set is our test set** and is
**stratified-split out of the training data** so all 251 classes are represented.


## 1. Setup — paths, seed, device

In [ ]:
import sys, os, json, shutil
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

# Make `codes` importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from codes import config as C
from codes import utils as U
from codes import data_handler as dh
from codes import model as M
from codes import loss_function as L
from codes import train as T
from codes import evaluate as E
from codes import self_supervised as S
from codes import hyperparameter_tuning as HT

U.set_seed(C.SEED)
device = U.get_device()
U.create_directories(C.MODELS_DIR, C.RESULTS_DIR)
print("Device:", device)
print(f"NUM_WORKERS={C.NUM_WORKERS} (device-aware, auto-resolved -- see the "
      f"[num_workers] log line above / config.py)")
print(f"TUNE_MODE={C.TUNE_MODE} (strategy={C.TUNE_STRATEGY}, "
      f"n_configs={C.TUNE_N_RANDOM_CONFIGS}, probe_epochs={C.TUNE_PROBE_EPOCHS}) "
      f"-- set config.TUNE_MODE='full' for the real report run, NOT 'fast_dev'")

# Idempotency flags (config.py is the single source of truth for the defaults;
# these notebook-local names are what every "SKIPPING/RUNNING" cell below checks).
# FORCE_RECOMPUTE_DATA -> ignore cached outlier/cleaning outputs and rescan.
# FORCE_RETRAIN         -> ignore cached model checkpoints/hparams and retrain.
# To force a rerun of ONE specific stage instead of everything, just delete
# that stage's output file(s) under results/ or models/ rather than flipping
# these -- each guarded cell prints exactly which path(s) it checked.
FORCE_RECOMPUTE_DATA = C.FORCE_RECOMPUTE_DATA
FORCE_RETRAIN = C.FORCE_RETRAIN
print(f"FORCE_RECOMPUTE_DATA={FORCE_RECOMPUTE_DATA} | FORCE_RETRAIN={FORCE_RETRAIN}")

**Idempotency note.** This notebook is safe to re-run top to bottom: every expensive stage (outlier scan, cleaned-manifest build, hyperparameter search, model/SSL training) checks for its expected output file(s) first and prints `SKIPPING: ...` + reloads them instead of recomputing, or `RUNNING: ...` when it actually (re)computes. To force a clean rerun of *everything* in a given category, flip `FORCE_RECOMPUTE_DATA = True` (data/outlier stages) and/or `FORCE_RETRAIN = True` (model/SSL training) above. To force a rerun of just ONE stage, simply delete that stage's output file(s) under `results/` or `models/` instead.

## 2. Load dataset manifest
Load the labels CSV and normalise it to `image_id` / `label`. We defer the train/validation split until **after**
outlier handling, so the split is built from the cleaned manifest.

In [ ]:
# Named raw_train_df (not train_df) deliberately: cell 16 below builds the
# actual training split and assigns THAT to train_df. Keeping the raw,
# pre-outlier-removal manifest under its own name means cells that need it
# (this one, class distribution, the outlier audit, before/after chart) give
# the same answer regardless of what order cells are rerun in.
raw_train_df = dh.build_dataframe(C.TRAIN_CSV)
test_df = dh.build_dataframe(C.TEST_CSV)
print(f"Loaded {len(raw_train_df):,} training images across {raw_train_df['label'].nunique()} classes")
print(f"Loaded {len(test_df):,} test images across {test_df['label'].nunique()} classes")
assert raw_train_df['label'].nunique() == C.NUM_CLASSES, "Dataset must keep all 251 classes"
assert test_df['label'].nunique() == C.NUM_CLASSES, "Dataset must keep all 251 classes"
class_names = dh.load_class_names(raw_train_df, C.NUM_CLASSES)

In [ ]:
# Class distribution
label_col = [c for c in raw_train_df.columns if c.lower() in
             ("label","class","class_id","category","target")][0]
class_counts = raw_train_df[label_col].value_counts()

print(f"Total images  : {len(raw_train_df):,}")
print(f"Unique classes: {raw_train_df[label_col].nunique()}")
print(f"\nSmallest 5 classes:\n{class_counts.tail()}")
print(f"\nLargest 5 classes:\n{class_counts.head()}")

counts = sorted(class_counts.values, reverse=True)

fig, ax = plt.subplots(figsize=(14, 5), facecolor="#1a1a2e")
ax.set_facecolor("#12122a")

ax.bar(range(len(counts)), counts, color="#7b7baa", edgecolor="white", linewidth=1.0, width=0.9 )

ax.set_xlabel("Class rank", color="white")
ax.set_ylabel("Image count", color="white")
ax.set_title( "Images per class (sorted)",color="white",fontweight="bold")

ax.tick_params(colors="white")

# grid like second plot
ax.grid(True,color="white",alpha=0.6,linestyle="-")

ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color("white")

plt.tight_layout()

plt.savefig(os.path.join(C.RESULTS_DIR, "class_distribution.png"),
            dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("↑ saved class_distribution.png")

## 3. Outlier handling (3-stage pipeline)
A 3-stage audit tailored to food imagery, run inline:

1. **Stage 1 — integrity** (auto-removed): missing / corrupt / undersized / near-black / near-white / low-contrast.
2. **Stage 2 — pixel statistics** (flagged for review): colour / texture / uniqueness bounds catch likely non-food images.
3. **Stage 3 — embeddings** (flagged for review): ResNet-50 features + global & per-class IsolationForest catch images
   that look unlike their own class (mislabels, off-topic shots).

Stages 2–3 only *flag* — after a quick visual review you confirm removals and write a cleaned manifest, which the
rest of the notebook then trains on. Set `SKIP_STAGE3 = True` for a fast CPU-only first pass.

In [ ]:
from codes.outlier_handler import (
    run_outlier_pipeline, outlier_stage_outputs_exist, load_cached_outlier_outputs,
    visualize_flagged_images, plot_anomaly_score_distribution, apply_review_decisions,
    plot_before_after_class_counts, N_FAILS_DEFAULT,
)

PER_CLASS_THR = -0.15   # lower -> more Stage-3 flags
GLOBAL_THR    = -0.12
SKIP_STAGE3   = False    # set True for a quick CPU-only first pass

# Idempotent: the 3-stage scan opens every image (Stage 1/2) and runs a
# ResNet-50 forward pass over the whole dataset (Stage 3) -- expensive to
# redo on every notebook rerun, so check for the existing outputs FIRST.
if not FORCE_RECOMPUTE_DATA and outlier_stage_outputs_exist(str(C.RESULTS_DIR), skip_stage3=SKIP_STAGE3):
    print(f"SKIPPING: outlier-pipeline outputs already exist under {C.RESULTS_DIR} -- loading them "
          f"(delete them, or set FORCE_RECOMPUTE_DATA=True, to force a rerun).")
    (
        df_clean,         # Stage-1 cleaned manifest (integrity failures removed)
        stats_df,         # Stage-2 pixel statistics (all images)
        flagged2,         # Stage-2 flagged (review)
        feats,            # Stage-3 PCA embeddings
        emb_ids,          # Stage-3 image IDs
        global_scores,    # Stage-3 global IF scores
        per_class_scores, # Stage-3 per-class IF scores
        flagged3,         # Stage-3 flagged (review)
    ) = load_cached_outlier_outputs(str(C.TRAIN_CSV), str(C.RESULTS_DIR), skip_stage3=SKIP_STAGE3)
else:
    print("RUNNING: computing the 3-stage outlier audit (no cached outputs found, or FORCE_RECOMPUTE_DATA=True).")
    (
        df_clean, stats_df, flagged2, feats, emb_ids, global_scores, per_class_scores, flagged3,
    ) = run_outlier_pipeline(
        csv_path      = str(C.TRAIN_CSV),
        img_dir       = str(C.IMAGE_DIR),
        out_dir       = str(C.RESULTS_DIR),
        device        = str(device),
        global_thr    = GLOBAL_THR,
        per_class_thr = PER_CLASS_THR,
        n_fails_required = N_FAILS_DEFAULT,
        skip_stage3   = SKIP_STAGE3,
    )

In [ ]:
# Stage 1 -- integrity (auto-removed, no review needed)
s1_log = C.RESULTS_DIR / 'removed_stage1_integrity.csv'
if s1_log.exists():
    s1_df = pd.read_csv(s1_log)
    print(f'Auto-removed: {len(s1_df):,}')
    display(s1_df['reason'].str.split('(').str[0].value_counts().to_frame('count'))
else:
    print('All images passed Stage 1 (or the run that produced this cache found none) OK')
print(f'Surviving after Stage 1: {len(df_clean):,}  |  classes: {df_clean["label"].nunique()}')

In [ ]:
# Stage 2 — review flagged (likely non-food by pixel statistics)
print(f'Stage 2 flagged: {len(flagged2):,} / {len(stats_df):,}')
if not flagged2.empty:
    cols = [c for c in ['image_id','label','n_fails','fail_reasons'] if c in flagged2.columns]
    display(flagged2[cols].head(20))
    visualize_flagged_images(flagged2, str(C.IMAGE_DIR), title='Stage 2 — non-food (pixel stats)',
                             save_path=str(C.RESULTS_DIR / 'stage2_flagged_grid.png'))

In [ ]:
# Stage 3 — review embedding outliers
if not SKIP_STAGE3 and len(per_class_scores) > 0:
    print(f'Stage 3 flagged: {len(flagged3):,}')
    plot_anomaly_score_distribution(per_class_scores, score_thr=PER_CLASS_THR,
        title='Stage 3 — per-class IsolationForest scores',
        save_path=str(C.RESULTS_DIR / 'stage3_anomaly_dist.png'))
    if not flagged3.empty:
        visualize_flagged_images(flagged3, str(C.IMAGE_DIR), title='Stage 3 — embedding outliers',
                                 save_path=str(C.RESULTS_DIR / 'stage3_flagged_grid.png'))
else:
    print('Stage 3 skipped.')

In [ ]:
# ACTION: after visual review, confirm removals -> cleaned manifest.
# The review CSVs (results/review_stage2_*.csv, review_stage3_*.csv) default every row to "remove";
# edit them to "keep" any false positives, save as confirmed_remove_stage{2,3}.csv, then rerun this cell.
confirmed = [p for p in (C.RESULTS_DIR/'confirmed_remove_stage2.csv',
                         C.RESULTS_DIR/'confirmed_remove_stage3.csv') if p.exists()]
clean_deps = confirmed + [C.RESULTS_DIR/'review_stage2_nonfood_pixelstats.csv',
                          C.RESULTS_DIR/'review_stage3_nonfood_embedding.csv',
                          C.RESULTS_DIR/'removed_stage1_integrity.csv']

# Idempotent: only rebuild train_labels_clean.csv if it's missing, stale
# w.r.t. its stage1/2/3 dependencies, or FORCE_RECOMPUTE_DATA=True.
if not FORCE_RECOMPUTE_DATA and U.is_fresh(C.CLEAN_CSV, *clean_deps):
    print(f"SKIPPING: {C.CLEAN_CSV} is up to date with its stage1/2/3 inputs -- loading it "
          f"(delete it, or set FORCE_RECOMPUTE_DATA=True, to force a rebuild).")
    df_final = pd.read_csv(C.CLEAN_CSV)
else:
    print("RUNNING: (re)building the cleaned manifest from the review decisions.")
    if confirmed:
        df_final = apply_review_decisions(df_clean, [str(p) for p in confirmed],
                                          output_csv=str(C.CLEAN_CSV))
    else:
        # No confirmed removals yet -> proceed with the Stage-1 cleaned manifest.
        df_final = df_clean.copy()
        df_final.to_csv(C.CLEAN_CSV, index=False)

print(f'Training manifest: {len(df_final):,} images across {df_final["label"].nunique()} classes')
assert df_final['label'].nunique() == C.NUM_CLASSES, 'All 251 classes must survive cleaning'


In [ ]:
from codes.outlier_handler import audit_summary_report

# Per-class audit trail across all 3 stages: confirms the cleaning pipeline
# did not gut any class (especially the ~34-image tail classes) below a
# usable minimum. Written to results/outlier_summary_by_class.csv.
audit_path = C.RESULTS_DIR / "outlier_summary_by_class.csv"
if not FORCE_RECOMPUTE_DATA and audit_path.exists():
    print(f"SKIPPING: {audit_path} already exists -- loading it "
          f"(delete it, or set FORCE_RECOMPUTE_DATA=True, to force a rebuild).")
    outlier_summary = pd.read_csv(audit_path)
else:
    print("RUNNING: building the per-class outlier audit summary.")
    outlier_summary = audit_summary_report(
        raw_df=raw_train_df, stage1_df=df_clean, flagged2_df=flagged2, flagged3_df=flagged3,
        final_df=df_final, num_classes=C.NUM_CLASSES, out_dir=str(C.RESULTS_DIR),
        min_remaining=C.OUTLIER_MIN_CLASS_REMAINING,
    )

In [ ]:
# Before/after outlier-removal per-class comparison: the clearest
# single figure for "what did the 3-stage audit actually remove", complementing
# the stage-by-stage flagged-image grids above.
ba_path = C.RESULTS_DIR / "outlier_before_after.png"
plot_before_after_class_counts(
    raw_train_df, df_final, num_classes=C.NUM_CLASSES,
    save_path=str(ba_path),
)

## 3b. Preprocessing — sub-sample & stratified split
Optionally cap images-per-class for documented computational reasons (the **number of classes stays 251**), then build
the **stratified** train/validation split from the cleaned manifest. Preprocessing per image (resize to a common
`INPUT_SIZE` square, keep RGB, ImageNet normalisation) is applied inside the dataset pipelines.

In [ ]:
# Cap images per class only (classes preserved) -- a config-driven, documented
# computational-cost decision (config.MAX_IMAGES_PER_CLASS; None = full data).
# Reads the CLEANED manifest (post-outlier-removal), not the raw CSV.
df = pd.read_csv(C.CLEAN_CSV)

df = dh.cap_images_per_class(df, C.MAX_IMAGES_PER_CLASS, seed=C.SEED)
print(f"After sampling: {len(df):,} images across {df['label'].nunique()} classes "
      f"(cap={C.MAX_IMAGES_PER_CLASS or 'none -- full data'})")

# Validation = our test set, stratified out of train.
train_df, val_df = dh.stratified_split(df, val_split=C.VAL_SPLIT, seed=C.SEED)
print(f"Train: {len(train_df):,}  |  Val(=test): {len(val_df):,}")
print(f"Classes present in val: {val_df['label'].nunique()} / {C.NUM_CLASSES}")

# Class-distribution sanity plot. Cheap to redraw, but guard the SAVE so a
# rerun doesn't keep touching the file's timestamp for no reason.
split_plot_path = C.RESULTS_DIR / "train_split_class_distribution.png"
counts = train_df['label'].value_counts().sort_index()
plt.figure(figsize=(14,3))
plt.bar(counts.index, counts.values, width=1.0)
plt.title("Training images per class (251 classes)")
plt.xlabel("class id"); plt.ylabel("count"); plt.tight_layout()
if FORCE_RECOMPUTE_DATA or not split_plot_path.exists():
    plt.savefig(split_plot_path, dpi=120, bbox_inches="tight")
    print(f"  -> saved: {split_plot_path}")
else:
    print(f"  -> SKIPPING save: {split_plot_path} already exists (set FORCE_RECOMPUTE_DATA=True to overwrite)")
plt.show()
print("min/median/max per class:", counts.min(), int(counts.median()), counts.max())

In [ ]:
# Tail classes: smallest TAIL_CLASS_FRACTION of classes by image count. Shared
# definition reused for (a) class-aware augmentation boost (next cell) and
# (b) head/tail per-class metric reporting later, so both stay consistent.
tail_classes = dh.compute_tail_classes(train_df, C.NUM_CLASSES, C.TAIL_CLASS_FRACTION)
print(f"{len(tail_classes)} tail classes (of {C.NUM_CLASSES}), "
      f"e.g. {sorted(list(tail_classes))[:10]}...")


## Output files reference
| File | Stage | Purpose |
|---|---|---|
| `removed_stage1_integrity.csv` | 1 | Auto-removed images (no review needed) |
| `review_stage2_nonfood_pixelstats.csv` | 2 | All flagged images to review |
| `review_stage3_nonfood_embedding.csv` | 3 | All embedding-flagged images to review |
| `train_labels_clean.csv` | Final | Cleaned manifest — use for training |
| `class_distribution.png` | EDA | Image count per class |
| `stage2_flagged_grid.png` | 2 | Thumbnail grid of flagged images |
| `stage3_perclass_score_dist.png` | 3 | Per-class IF score histogram |
| `stage3_flagged_grid.png` | 3 | Thumbnail grid of flagged images |

All files are written to `../results/`.


## 4. DataLoaders
Class imbalance is handled via weighted CE / focal loss (see `loss_function.py`). A single uniform
augmentation pipeline is applied across all classes. Pick ONE class-frequency correction:
weighted sampler XOR loss weights (here we use loss weights via `config.LOSS_TYPE`).

In [ ]:
# Datasets -- tail classes get a boosted augmentation intensity (class-aware
# augmentation, config.USE_TAIL_AWARE_AUGMENTATION); every other class uses
# the uniform config.AUGMENTATION_INTENSITY. Validation uses no augmentation
# (deterministic evaluation).
aug_tail_classes = tail_classes if C.USE_TAIL_AWARE_AUGMENTATION else None
train_ds = dh.FoodDataset(train_df, C.IMAGE_DIR, augment=True, image_size=C.INPUT_SIZE,
                          intensity=C.AUGMENTATION_INTENSITY,
                          tail_classes=aug_tail_classes, tail_boost=C.TAIL_AUG_BOOST)
val_ds   = dh.FoodDataset(val_df,   C.IMAGE_DIR, augment=False, image_size=C.INPUT_SIZE)
test_ds  = dh.FoodDataset(test_df,  C.IMAGE_DIR, augment=False, image_size=C.INPUT_SIZE)

# Pick ONE imbalance correction: weighted sampler OR loss weights (see config).
sampler = dh.build_weighted_sampler(train_df, C.NUM_CLASSES) if C.USE_WEIGHTED_SAMPLER else None
train_loader = DataLoader(train_ds, batch_size=C.BATCH_SIZE, shuffle=(sampler is None),
                          sampler=sampler, num_workers=C.NUM_WORKERS, pin_memory=False,
                          **dh.loader_kwargs(C.NUM_WORKERS))
val_loader   = DataLoader(val_ds,   batch_size=C.BATCH_SIZE, shuffle=False,
                          num_workers=C.NUM_WORKERS, **dh.loader_kwargs(C.NUM_WORKERS))

print("Train images:", len(train_ds), "| Val images:", len(val_ds))


## 5. Custom CNN — architecture & parameter budget
Both custom architectures are verified against the **< 10 M parameter** budget before any
training: **FoodNet30** (`codes.model` registry key `"foodnet30"`, 30 Conv2d layers, residual
DWS + SE) and **FoodNet46** (registry key `"foodnet46"`, 46 Conv2d layers, MBConv
inverted-residual + SE + DropPath). Section 6 runs the full tune -> retrain -> evaluate
pipeline for BOTH and picks the overall winner; SSL (Section 7) then pretrains only that
winning architecture.

In [ ]:
for arch_name in M.MODEL_REGISTRY:
    m = M.build_model(arch_name, num_classes=C.NUM_CLASSES, dropout=C.DROPOUT, width_mult=C.WIDTH_MULT)
    info = m.model_info()
    print(info)
    assert info['under_10M'], f"{arch_name} exceeds the 10M parameter budget!"
    U.assert_param_budget(m)
    del m

## 6. Task A — Supervised Learning: tune, select, retrain (both architectures)

`run_supervised_pipeline` (defined below) tunes, retrains, and evaluates ONE architecture
end-to-end: Phase A/B searches a capped subset (`config.TUNE_STRATEGY`, successive halving by
default, ranked by validation macro-F1), with a LOCAL early stop inside every candidate's own
probe once it plateaus (`config.TUNE_EARLY_STOP_PATIENCE` -- independent of `Trainer`'s own
`PATIENCE`, which only governs the Phase C full retrain below). Phase C then retrains the
winning config on the FULL dataset for the full epoch budget.

It is idempotent: if `models/<arch>/best_model.pth` and
`results/<arch>_best_hparams.json` already exist, it loads them and skips straight to
evaluation, unless `FORCE_RETRAIN=True`.

We call it once per architecture (`"foodnet30"`, `"foodnet46"` -- the actual
`codes.model.MODEL_REGISTRY` keys; note these are lowercase, unlike the PascalCase
`FoodNet30`/`FoodNet46` class names) so the two runs are directly comparable and never
overwrite each other's outputs.

In [ ]:
# Phase A/B tuning loaders: a documented capped subset (config.TUNE_SUBSET_IMAGES_PER_CLASS),
# shared by BOTH architectures' hyperparameter search below.
tune_df = dh.cap_images_per_class(df_final, C.TUNE_SUBSET_IMAGES_PER_CLASS, seed=C.SEED)
tune_train_df, tune_val_df = dh.stratified_split(tune_df, val_split=C.VAL_SPLIT, seed=C.SEED)

tune_train_loader = DataLoader(
    dh.FoodDataset(tune_train_df, C.IMAGE_DIR, augment=True, image_size=C.INPUT_SIZE,
                  intensity=C.AUGMENTATION_INTENSITY),
    batch_size=C.BATCH_SIZE, shuffle=True, num_workers=C.NUM_WORKERS,
    **dh.loader_kwargs(C.NUM_WORKERS))
tune_val_loader = DataLoader(
    dh.FoodDataset(tune_val_df, C.IMAGE_DIR, augment=False, image_size=C.INPUT_SIZE),
    batch_size=C.BATCH_SIZE, num_workers=C.NUM_WORKERS, **dh.loader_kwargs(C.NUM_WORKERS))

In [ ]:
def run_supervised_pipeline(model_name: str, class_names, tail_classes,
                            train_loader, val_loader, tune_train_loader, tune_val_loader,
                            device, force_retrain: bool | None = None) -> dict:
    """Tune, retrain, evaluate ONE architecture end-to-end.

    Idempotent: if MODELS_DIR/<model_name>/best_model.pth and
    RESULTS_DIR/f"{model_name}_best_hparams.json" already exist and
    force_retrain is False, loads them and skips tuning/training entirely,
    going straight to evaluation.

    Returns a dict with: model, trainer, history, metrics, per_class_df,
    tail_head_summary, best_config, model_dir, params_M, true_labels, predictions.
    """
    force_retrain = C.FORCE_RETRAIN if force_retrain is None else force_retrain
    model_dir = Path(C.MODELS_DIR) / model_name
    ckpt_path = model_dir / "best_model.pth"
    hparams_path = Path(C.RESULTS_DIR) / f"{model_name}_best_hparams.json"
    history_path = Path(C.RESULTS_DIR) / f"{model_name}_history.json"

    if not force_retrain and ckpt_path.exists() and hparams_path.exists():
        print(f"[{model_name}] SKIPPING: found existing trained model + hparams "
              f"(delete {ckpt_path} / {hparams_path}, or set FORCE_RETRAIN=True, to force a rerun).")
        with open(hparams_path) as f:
            saved = json.load(f)
        best_cfg = saved["config"]
        history = None
        if history_path.exists():
            with open(history_path) as f:
                history = json.load(f)
        else:
            print(f"[{model_name}] No saved training history found ({history_path}) "
                  f"-- curve plotting below will be skipped for this architecture.")
    else:
        print(f"[{model_name}] RUNNING: hyperparameter search + full retrain from scratch.")
        # ---- Phase A: cheap search on the capped subset (LR/optimizer/wd/dropout) ----
        sl_grid = HT.default_sl_grid()
        sl_grid["model_name"] = [model_name]   # override default_sl_grid's hardcoded model
        sl_best, sl_results = HT.tune_supervised(
            tune_train_loader, tune_val_loader, device=device, grid=sl_grid,
            probe_epochs=C.TUNE_PROBE_EPOCHS, num_classes=C.NUM_CLASSES,
            strategy=C.TUNE_STRATEGY, selection_metric=C.TUNE_SELECTION_METRIC,
            n_random_configs=C.TUNE_N_RANDOM_CONFIGS, use_weighted_sampler=C.USE_WEIGHTED_SAMPLER,
            data_subset="capped", early_stop_patience=C.TUNE_EARLY_STOP_PATIENCE, seed=C.SEED,
            csv_path=str(C.RESULTS_DIR / f"{model_name}_sl_tuning_lr_wd_dropout.csv"))

        # Phase A continued: imbalance axis, holding the winning LR/WD/dropout/model config fixed.
        imb_grid = HT.grid_with_overrides(sl_best.config, HT.imbalance_grid())
        imb_best, imb_results = HT.tune_supervised(
            tune_train_loader, tune_val_loader, device=device, grid=imb_grid,
            probe_epochs=C.TUNE_PROBE_EPOCHS, num_classes=C.NUM_CLASSES, strategy="grid",
            selection_metric=C.TUNE_SELECTION_METRIC, use_weighted_sampler=C.USE_WEIGHTED_SAMPLER,
            data_subset="capped", early_stop_patience=C.TUNE_EARLY_STOP_PATIENCE, seed=C.SEED,
            csv_path=str(C.RESULTS_DIR / f"{model_name}_sl_tuning_imbalance.csv"))

        HT.write_tuning_summary(imb_best, sl_results + imb_results,
                                str(C.RESULTS_DIR / f"{model_name}_sl_tuning_summary.md"), n=5,
                                selection_metric=C.TUNE_SELECTION_METRIC)
        best_cfg = imb_best.config
        print(f"[{model_name}] Winning SL config (Phase A/B):", best_cfg)

        # ---- Phase C: full retrain on the full dataset, the only run that goes the full distance ----
        phase_c_model = M.build_model(best_cfg["model_name"], num_classes=C.NUM_CLASSES,
                                      dropout=best_cfg.get("dropout", C.DROPOUT),
                                      width_mult=best_cfg.get("width_mult", C.WIDTH_MULT))
        U.assert_param_budget(phase_c_model)

        class_weights = None if C.USE_WEIGHTED_SAMPLER else dh.compute_class_weights(
            train_loader.dataset.df, C.NUM_CLASSES,
            scheme=best_cfg.get("class_weight_scheme", C.CLASS_WEIGHT_SCHEME)).to(device)
        dh.check_single_imbalance_correction(C.USE_WEIGHTED_SAMPLER, class_weights)
        criterion = L.build_criterion(best_cfg.get("loss_type", C.LOSS_TYPE), class_weights=class_weights,
                                      gamma=C.FOCAL_GAMMA, label_smoothing=best_cfg.get("label_smoothing", C.LABEL_SMOOTHING))

        trainer = T.Trainer(phase_c_model, device, criterion=criterion,
                            learning_rate=best_cfg.get("lr", C.LEARNING_RATE),
                            weight_decay=best_cfg.get("weight_decay", C.WEIGHT_DECAY),
                            mix_method=C.MIX_METHOD, mixup_alpha=C.MIXUP_ALPHA, cutmix_alpha=C.CUTMIX_ALPHA)
        history = trainer.train(train_loader, val_loader,
                                num_epochs=C.NUM_EPOCHS, patience=C.PATIENCE,
                                model_save_dir=str(C.MODELS_DIR), run_name=model_name,
                                warmup_epochs=C.WARMUP_EPOCHS,
                                log_per_class_every=C.LOG_PER_CLASS_EVERY,
                                class_names=list(class_names.values()), tail_classes=tail_classes,
                                results_dir=str(C.RESULTS_DIR))

        history_path.parent.mkdir(parents=True, exist_ok=True)
        with open(history_path, "w") as f:
            json.dump(history, f)

        # Save the winning hyperparameters + selection metric value.
        selection_value = imb_best.val_f1_macro if C.TUNE_SELECTION_METRIC == "f1_macro" else imb_best.val_accuracy
        hparams_path.parent.mkdir(parents=True, exist_ok=True)
        with open(hparams_path, "w") as f:
            json.dump({"config": best_cfg, "selection_metric": C.TUNE_SELECTION_METRIC,
                       "selection_metric_value": selection_value}, f, indent=2)
        print(f"[{model_name}] Saved best hyperparameters -> {hparams_path}")

    # ---- Build a fresh model + trainer, then load the best checkpoint ----
    # This runs whether we just trained (reload the BEST epoch, which may not be
    # the last epoch trained) or loaded from cache (nothing to build otherwise).
    model = M.build_model(best_cfg["model_name"], num_classes=C.NUM_CLASSES,
                          dropout=best_cfg.get("dropout", C.DROPOUT),
                          width_mult=best_cfg.get("width_mult", C.WIDTH_MULT))
    trainer = T.Trainer(model, device, mix_method=C.MIX_METHOD,
                        mixup_alpha=C.MIXUP_ALPHA, cutmix_alpha=C.CUTMIX_ALPHA)
    trainer.load_checkpoint(ckpt_path, weights_only=True)

    # ---- Evaluation (always runs) ----
    evaluator = E.Evaluator(num_classes=C.NUM_CLASSES, class_names=list(class_names.values()), device=device)
    eval_out = evaluator.evaluate(model, val_loader)
    metrics = eval_out["metrics"]
    metrics["top5_accuracy"] = E.Evaluator.top_k_accuracy(model, val_loader, device, k=5)

    per_class_df = evaluator.per_class_metrics(eval_out["true_labels"], eval_out["predictions"])
    per_class_df.to_csv(C.RESULTS_DIR / f"{model_name}_sl_per_class_final.csv", index=False)
    tail_head_summary = evaluator.head_vs_tail_summary(per_class_df, tail_classes=tail_classes)
    print(f"[{model_name}] tail vs head:", tail_head_summary)

    T.log_run(f"supervised_{model_name}_final", best_cfg,
             {**metrics, **tail_head_summary}, str(C.RESULTS_DIR / "experiment_log.csv"))

    # Per-architecture loss/accuracy curves.
    if history is not None:
        E.plot_training_curves(history, save_path=str(C.RESULTS_DIR / f"{model_name}_sl_curves.png"))

    return {
        "model": model, "trainer": trainer, "history": history, "metrics": metrics,
        "per_class_df": per_class_df, "tail_head_summary": tail_head_summary,
        "best_config": best_cfg, "model_dir": model_dir,
        "params_M": model.model_info()["total_params_M"],
        "true_labels": eval_out["true_labels"], "predictions": eval_out["predictions"],
    }

In [ ]:
# Run the full tune -> retrain -> evaluate pipeline for BOTH custom architectures.
# Every output filename is prefixed with the architecture name so the
# two runs never overwrite each other (models/<arch>/best_model.pth,
# results/<arch>_sl_tuning_*.csv, results/<arch>_sl_curves.png, ...).
results_by_model = {}
for arch in M.MODEL_REGISTRY:
    results_by_model[arch] = run_supervised_pipeline(
        model_name=arch, class_names=class_names, tail_classes=tail_classes,
        train_loader=train_loader, val_loader=val_loader,
        tune_train_loader=tune_train_loader, tune_val_loader=tune_val_loader,
        device=device,
    )

In [ ]:
# Pick the overall best supervised architecture by the configured
# selection metric (config.TUNE_SELECTION_METRIC), and save a comparison table.
overall_best_sl = max(results_by_model, key=lambda name: results_by_model[name]["metrics"][C.TUNE_SELECTION_METRIC])
print(f"Overall best supervised architecture: {overall_best_sl}")

sl_comparison_rows = [
    {
        "architecture": arch,
        "params_M": res["params_M"],
        C.TUNE_SELECTION_METRIC: res["metrics"][C.TUNE_SELECTION_METRIC],
        "accuracy": res["metrics"]["accuracy"],
        "top5_accuracy": res["metrics"]["top5_accuracy"],
        "is_overall_best": arch == overall_best_sl,
    }
    for arch, res in results_by_model.items()
]
sl_comparison_df = pd.DataFrame(sl_comparison_rows).sort_values(C.TUNE_SELECTION_METRIC, ascending=False)
display(sl_comparison_df)
sl_comparison_df.to_csv(C.RESULTS_DIR / "sl_model_comparison.csv", index=False)

# Kept under the ORIGINAL variable names (sl_result, evaluator, model) so the
# rest of the notebook (confusion matrix, Grad-CAM, SL-vs-SSL comparison)
# reads exactly like a single-model run, just pointed at the overall winner.
sl_result = results_by_model[overall_best_sl]
model = sl_result["model"]
evaluator = E.Evaluator(num_classes=C.NUM_CLASSES, class_names=list(class_names.values()), device=device)

In [ ]:
# Overlay both architectures' train/val loss curves on one figure so
# their convergence behaviour can be compared at a glance.
fig, ax = plt.subplots(figsize=(8, 5))
for arch, res in results_by_model.items():
    if res["history"] is None:
        print(f"[{arch}] no saved history available -- skipped in the overlay "
              f"(this happens if its checkpoint predates history saving; delete "
              f"models/{arch}/ and results/{arch}_history.json to force a retrain that saves one).")
        continue
    ax.plot(res["history"]["train_loss"], label=f"{arch} train", linestyle="--")
    ax.plot(res["history"]["val_loss"], label=f"{arch} val")
ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.set_title(f"Supervised loss comparison: {' vs '.join(results_by_model.keys())}")
ax.legend()
plt.tight_layout()
plt.savefig(C.RESULTS_DIR / "sl_loss_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
evaluator.plot_confusion_matrix(sl_result['true_labels'], sl_result['predictions'],
                                normalize=True,
                                save_path=str(C.RESULTS_DIR / f"{overall_best_sl}_sl_confusion.png"))

## 7. Task B — Self-Supervised Learning: tune each method, then compare

The SSL paradigm pretrains the same backbone without labels, freezes it, extracts features, and
classifies them with a traditional classifier. **SSL always pretrains the architecture
that WON the supervised comparison above (`overall_best_sl`)** — NOT necessarily the same
hyperparameters, since SSL pretraining has its own tuned LR/temperature/etc. (tuned separately
below). We tune the two pretext methods separately, because they consume different inputs:
SimCLR needs augmented image pairs, rotation needs single images. Each method is tuned with a
full grid, then retrained to convergence at its winning config (BOTH methods' curves are
saved even though only one is selected as the final SSL result), and the two are compared so the
better pretext method is selected.

In [ ]:
# Loaders for SSL. SimCLR consumes augmented pairs; rotation and feature
# extraction consume single images.
ssl_pair_loader = DataLoader(
    dh.SSLPairDataset(train_df, C.IMAGE_DIR, image_size=C.INPUT_SIZE),
    batch_size=C.SSL_BATCH_SIZE, shuffle=True, num_workers=C.NUM_WORKERS, drop_last=True,
    **dh.loader_kwargs(C.NUM_WORKERS))

ssl_image_loader = DataLoader(
    dh.FeatureExtractionDataset(train_df, C.IMAGE_DIR, image_size=C.INPUT_SIZE),
    batch_size=C.SSL_BATCH_SIZE, shuffle=True, num_workers=C.NUM_WORKERS, drop_last=True,
    **dh.loader_kwargs(C.NUM_WORKERS))

feat_train_loader = DataLoader(
    dh.FeatureExtractionDataset(train_df, C.IMAGE_DIR, image_size=C.INPUT_SIZE),
    batch_size=C.BATCH_SIZE, num_workers=C.NUM_WORKERS, **dh.loader_kwargs(C.NUM_WORKERS))
feat_val_loader = DataLoader(
    dh.FeatureExtractionDataset(val_df, C.IMAGE_DIR, image_size=C.INPUT_SIZE),
    batch_size=C.BATCH_SIZE, num_workers=C.NUM_WORKERS, **dh.loader_kwargs(C.NUM_WORKERS))

# SSL always pretrains the architecture that WON the supervised
# comparison; its own LR/temperature/etc. are tuned separately below,
# not copied from the supervised winner's hyperparameters.
ssl_grid = HT.default_ssl_grid()
ssl_grid["model_name"] = [overall_best_sl]
print(f"SSL will pretrain: {overall_best_sl} (the supervised winner)")

# Warn (don't silently reuse) if a PREVIOUS run's SSL checkpoint on disk was
# trained on a DIFFERENT architecture than the current supervised winner.
ssl_hparams_path = C.RESULTS_DIR / "ssl_best_hparams.json"
if ssl_hparams_path.exists():
    with open(ssl_hparams_path) as f:
        _prev_ssl = json.load(f)
    _prev_arch = _prev_ssl.get("config", {}).get("model_name")
    if _prev_arch and _prev_arch != overall_best_sl:
        print(f"WARNING: {ssl_hparams_path} records a PREVIOUS SSL run on architecture "
              f"'{_prev_arch}', but the current supervised winner is '{overall_best_sl}'. "
              f"The stale backbone at models/ssl_best/backbone.pth will be OVERWRITTEN "
              f"below (not silently reused) -- back it up first if you need to keep it.")

In [ ]:
def run_ssl_method_pipeline(method: str, cfg: dict, ssl_loader, feat_train_loader,
                            feat_val_loader, device, force_retrain: bool | None = None) -> dict:
    """Full (non-probe) SSL retrain for ONE pretext method at its tuned config.

    Idempotent: if MODELS_DIR/ssl_<method>/backbone.pth and
    RESULTS_DIR/ssl_<method>_hparams.json both exist and force_retrain is
    False, skips the (expensive) pretraining and reloads the backbone --
    only a cheap feature-extraction + classifier refit is redone, since that
    is needed to report metrics either way.

    Returns {"ssl_out", "history", "backbone_path", "metrics", "config"}.
    """
    force_retrain = C.FORCE_RETRAIN if force_retrain is None else force_retrain
    method_dir = Path(C.MODELS_DIR) / f"ssl_{method}"
    backbone_path = method_dir / "backbone.pth"
    hparams_path = Path(C.RESULTS_DIR) / f"ssl_{method}_hparams.json"
    history_path = Path(C.RESULTS_DIR) / f"ssl_{method}_history.json"

    if not force_retrain and backbone_path.exists() and hparams_path.exists():
        print(f"[ssl:{method}] SKIPPING pretraining: found existing backbone + hparams "
              f"(delete {backbone_path} / {hparams_path}, or set FORCE_RETRAIN=True, to force a rerun). "
              f"Still re-extracting features + refitting the classifier (cheap) to report metrics.")
        history = None
        if history_path.exists():
            with open(history_path) as f:
                history = json.load(f)
        backbone = M.build_model(cfg["model_name"], num_classes=C.NUM_CLASSES,
                                 width_mult=cfg.get("width_mult", C.WIDTH_MULT))
        backbone.load_state_dict(torch.load(backbone_path, map_location=device))
        backbone = backbone.to(device)
        Xtr, ytr = S.extract_features(backbone, feat_train_loader, device)
        Xva, yva = S.extract_features(backbone, feat_val_loader, device)
        clf = S.fit_traditional_classifier(Xtr, ytr, classifier=cfg.get("classifier", C.SSL_CLASSIFIER), seed=C.SEED)
        val_pred = clf.predict(Xva)
        ssl_out = {"ssl_history": history, "classifier": clf,
                   "train_features": Xtr, "train_labels": ytr,
                   "val_features": Xva, "val_labels": yva, "val_predictions": val_pred}
    else:
        print(f"[ssl:{method}] RUNNING: full pretraining ({C.SSL_EPOCHS} epochs) + classifier fit.")
        backbone = M.build_model(cfg["model_name"], num_classes=C.NUM_CLASSES,
                                 width_mult=cfg.get("width_mult", C.WIDTH_MULT))
        ssl_out = S.run_ssl_pipeline(
            backbone, ssl_loader, feat_train_loader, feat_val_loader, device,
            method=method, classifier=cfg.get("classifier", C.SSL_CLASSIFIER),
            epochs=C.SSL_EPOCHS, lr=cfg.get("lr", C.SSL_LEARNING_RATE),
            weight_decay=cfg.get("weight_decay", C.SSL_WEIGHT_DECAY),
            temperature=cfg.get("temperature", C.SSL_TEMPERATURE),
            projection_dim=cfg.get("projection_dim", C.SSL_PROJECTION_DIM),
            save_path=str(backbone_path), seed=C.SEED)
        history = ssl_out["ssl_history"]
        history_path.parent.mkdir(parents=True, exist_ok=True)
        with open(history_path, "w") as f:
            json.dump(history, f)
        hparams_path.parent.mkdir(parents=True, exist_ok=True)
        with open(hparams_path, "w") as f:
            json.dump({"config": cfg, "method": method}, f, indent=2)
        print(f"[ssl:{method}] Saved hyperparameters -> {hparams_path}")

    metrics = E.Evaluator.metrics_from_predictions(ssl_out["val_labels"], ssl_out["val_predictions"])
    clf = ssl_out["classifier"]
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(ssl_out["val_features"])
        topk = np.argsort(-proba, axis=1)[:, :5]
        yva = ssl_out["val_labels"]
        metrics["top5_accuracy"] = float(np.mean([yva[i] in topk[i] for i in range(len(yva))]))
    else:
        print(f"[ssl:{method}] classifier '{cfg.get('classifier', C.SSL_CLASSIFIER)}' has no "
              f"predict_proba -- skipping top-5 accuracy for this method.")

    return {"ssl_out": ssl_out, "history": history, "backbone_path": backbone_path,
           "metrics": metrics, "config": cfg}

In [ ]:
# Tune SimCLR and rotation separately with full grids. Each candidate uses a
# LOCAL early stop inside its own probe budget once it plateaus
# (config.SSL_TUNE_EARLY_STOP_PATIENCE).
simclr_best, simclr_results = HT.tune_ssl(
    ssl_pair_loader, feat_train_loader, feat_val_loader,
    method="simclr", device=device, grid=ssl_grid,
    probe_epochs=C.SSL_TUNE_PROBE_EPOCHS, num_classes=C.NUM_CLASSES,
    selection_metric=C.TUNE_SELECTION_METRIC,
    early_stop_patience=C.SSL_TUNE_EARLY_STOP_PATIENCE,
    csv_path=str(C.RESULTS_DIR / "ssl_simclr_tuning.csv"))

rotation_best, rotation_results = HT.tune_ssl(
    ssl_image_loader, feat_train_loader, feat_val_loader,
    method="rotation", device=device, grid=ssl_grid,
    probe_epochs=C.SSL_TUNE_PROBE_EPOCHS, num_classes=C.NUM_CLASSES,
    selection_metric=C.TUNE_SELECTION_METRIC,
    early_stop_patience=C.SSL_TUNE_EARLY_STOP_PATIENCE,
    csv_path=str(C.RESULTS_DIR / "ssl_rotation_tuning.csv"))

In [ ]:
# Full (non-probe) retrain of BOTH methods at their tuned config -- idempotent
# per method, and needed so BOTH get a real training-curve figure
# even though only the better one is used downstream. This means
# BOTH methods now get a full C.SSL_EPOCHS run (not just the eventual winner),
# which is more compute than the original single-winner-only retrain.
simclr_full = run_ssl_method_pipeline("simclr", simclr_best.config, ssl_pair_loader,
                                      feat_train_loader, feat_val_loader, device)
rotation_full = run_ssl_method_pipeline("rotation", rotation_best.config, ssl_image_loader,
                                        feat_train_loader, feat_val_loader, device)

for _method, _res in (("simclr", simclr_full), ("rotation", rotation_full)):
    if _res["history"] is not None:
        E.plot_ssl_curves(_res["history"], method=_method,
                          save_path=str(C.RESULTS_DIR / f"ssl_{_method}_curves.png"))
    else:
        print(f"[ssl:{_method}] no saved history to plot (its checkpoint predates history saving).")

In [ ]:
# Compare the two tuned SSL winners (by the tuning-phase probe metric, same
# selection rule used throughout) and name the better pretext method.
ssl_choice = HT.compare_ssl_methods(simclr_best, rotation_best, selection_metric=C.TUNE_SELECTION_METRIC)
winner = ssl_choice["winner"]
print("Selected SSL method:", winner)

winner_full = simclr_full if winner == "simclr" else rotation_full
best_ssl_cfg = winner_full["config"]
ssl_out = winner_full["ssl_out"]

# Copy the winning method's confirmed backbone to the canonical
# models/ssl_best/backbone.pth path, and save its hyperparameters + selection
# metric value.
ssl_best_dir = Path(C.MODELS_DIR) / "ssl_best"
ssl_best_dir.mkdir(parents=True, exist_ok=True)
shutil.copyfile(winner_full["backbone_path"], ssl_best_dir / "backbone.pth")
print(f"[ssl] Copied winning backbone ({winner}) -> {ssl_best_dir / 'backbone.pth'}")

# The traditional read-out classifier (fit on frozen features) is never
# persisted by self_supervised.run_ssl_pipeline -- only kept in memory. Save
# it here so the Streamlit demo can load the SSL paradigm WITHOUT
# re-extracting features / re-fitting it.
import joblib
joblib.dump(ssl_out["classifier"], ssl_best_dir / "classifier.joblib")
print(f"[ssl] Saved downstream classifier -> {ssl_best_dir / 'classifier.joblib'}")

ssl_selection_value = winner_full["metrics"].get(C.TUNE_SELECTION_METRIC, winner_full["metrics"]["accuracy"])
with open(C.RESULTS_DIR / "ssl_best_hparams.json", "w") as f:
    json.dump({"method": winner, "config": best_ssl_cfg,
               "selection_metric": C.TUNE_SELECTION_METRIC,
               "selection_metric_value": ssl_selection_value}, f, indent=2)
print(f"[ssl] Saved best hyperparameters -> {C.RESULTS_DIR / 'ssl_best_hparams.json'}")

In [ ]:
ssl_metrics = winner_full["metrics"]
print(f"SSL ({winner}) metrics:")
for k, v in ssl_metrics.items():
    print(f"  {k:20s}: {v:.4f}")

In [ ]:
# Per-class report + head/tail breakdown for SSL, using the SAME Evaluator /
# tail_classes definition as the SL branch above, for a fair SL-vs-SSL comparison.
ssl_per_class = evaluator.per_class_metrics(ssl_out["val_labels"], ssl_out["val_predictions"])
ssl_per_class.to_csv(C.RESULTS_DIR / "ssl_per_class_final.csv", index=False)
ssl_tail_head = evaluator.head_vs_tail_summary(ssl_per_class, tail_classes=tail_classes)
print("SSL tail vs head:", ssl_tail_head)

T.log_run(f"ssl_{winner}_final", best_ssl_cfg,
         {**ssl_metrics, **ssl_tail_head}, str(C.RESULTS_DIR / "experiment_log.csv"))

## 8. Supervised vs Self-Supervised comparison

The supervised model (best of FoodNet30/FoodNet46) and the selected self-supervised model are
evaluated on the same validation split with the same metrics, so the comparison isolates the
single difference between them: whether the backbone was trained with labels or without.

In [ ]:
comparison = E.compare_paradigms(sl_result["metrics"], ssl_metrics)
comparison.insert(0, "sl_architecture", overall_best_sl)
comparison.insert(1, "ssl_method", winner)
print(f"Supervised winner: {overall_best_sl}  |  Self-supervised winner: {winner}")
display(comparison)
comparison.to_csv(C.RESULTS_DIR / "sl_vs_ssl_comparison.csv", index=False)

# Grouped bar chart of the headline metrics, SL vs SSL.
bar_metrics = [m for m in ("accuracy", "f1_macro", "top5_accuracy")
              if m in sl_result["metrics"] and m in ssl_metrics]
sl_vals = [sl_result["metrics"][m] for m in bar_metrics]
ssl_vals = [ssl_metrics[m] for m in bar_metrics]

x = np.arange(len(bar_metrics)); width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, sl_vals, width, label=f"Supervised ({overall_best_sl})")
ax.bar(x + width / 2, ssl_vals, width, label=f"Self-Supervised ({winner})")
ax.set_xticks(x); ax.set_xticklabels(bar_metrics)
ax.set_ylabel("score"); ax.set_title("Supervised vs Self-Supervised")
ax.legend()
plt.tight_layout()
plt.savefig(C.RESULTS_DIR / "sl_vs_ssl_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Notes on the hyperparameter search

Section 6 tunes BOTH architectures with `default_sl_grid()` (model_name overridden per
architecture) followed by `imbalance_grid()`; Section 7 tunes both SSL pretext methods with
`default_ssl_grid()` (model_name overridden to `overall_best_sl`). Every candidate probe uses a
local early stop once it plateaus (`config.TUNE_EARLY_STOP_PATIENCE` /
`config.SSL_TUNE_EARLY_STOP_PATIENCE`), recorded per-trial as `epochs_run` in the tuning CSVs.
To change the search space, edit `default_sl_grid`/`default_ssl_grid`/`imbalance_grid`, or pass a
custom grid dict to `tune_supervised`/`tune_ssl`.

In [ ]:
# The search spaces actually used above; inspect or override them as needed.
print("SL grid (LR/optimizer/wd/dropout), per architecture:", HT.default_sl_grid())
print("SL imbalance grid (own small grid, phase 2):", HT.imbalance_grid())
print("SL augmentation grid (optional phase, not auto-wired above):", HT.augmentation_grid())
print("SSL grid actually used (model_name overridden to the SL winner):", ssl_grid)
print(f"TUNE_EARLY_STOP_PATIENCE={C.TUNE_EARLY_STOP_PATIENCE} | "
      f"SSL_TUNE_EARLY_STOP_PATIENCE={C.SSL_TUNE_EARLY_STOP_PATIENCE}")

## 10. Grad-CAM explanations
Sanity-check that the supervised (overall-best) model attends to the food, not the plate/background.

In [ ]:
from codes.gradcam import GradCAM
model = sl_result["model"]   # the overall-best supervised model
gradcam = GradCAM(model)
images, labels = next(iter(val_loader))
x = images[:1].to(device)
cam = gradcam.generate(x)
# De-normalise for display
mean = np.array(dh.IMAGENET_MEAN); std = np.array(dh.IMAGENET_STD)
img = (x[0].cpu().numpy().transpose(1,2,0)*std + mean).clip(0,1)
fig, ax = plt.subplots(1,2, figsize=(8,4))
ax[0].imshow(img); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(gradcam.overlay(img, cam)); ax[1].set_title("Grad-CAM"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 11. Summary
- **Supervised**: both custom CNNs (< 10 M params, from scratch) tuned/retrained/evaluated end
  to end; `overall_best_sl` names the winner (Section 6).
- **Self-supervised**: the winning architecture pretrained with SimCLR and rotation (both fully
  retrained and curve-plotted), the better pretext method selected (`winner`).
- Same stratified train / val(=test) split, identical metrics (accuracy, precision, recall, F1,
  top-5 accuracy) for both paradigms.
- See the comparison table + grouped bar chart (Section 8) and the written report for discussion
  of results, limitations, and future work.